# Bildprojekt der 1. Klasse · Von Ziffern zu eigenen Bildklassen
**Doppellektionen 5–7.** Führt die Zellen der Reihe nach aus. Wir verwenden kleine Graustufenbilder (8×8 Pixel), damit das Training im Browser machbar bleibt.

**Forschungsfrage:** Was passiert, wenn ein Netz, das nur Ziffern gelernt hat, Buchstaben sieht? Können wir seine gelernten Merkmale für neue Klassen nutzen?

**Wichtig:** Eine Ziffernvorhersage für einen Buchstaben ist keine sinnvolle Buchstabenerkennung. Das Netz hat gar keine Buchstabenausgaben.

In [ ]:
import sys
if sys.platform == "emscripten":
    import pyodide_js
    await pyodide_js.loadPackage(["numpy","scipy","scikit-learn","matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
rng=np.random.default_rng(42)
digits=load_digits()
X=digits.data.astype(float)/16
y=digits.target
# Separate train and test images; test examples never enter training.
indices=rng.permutation(len(y)); train=indices[:1400]; test=indices[1400:]
digit_model=MLPClassifier(hidden_layer_sizes=(32,),max_iter=180,random_state=12,early_stopping=True)
digit_model.fit(X[train],y[train])
print("Ziffern: Testtrefferquote",round(accuracy_score(y[test],digit_model.predict(X[test])),2))
def zeigen(bilder,titel):
    fig,axs=plt.subplots(1,len(bilder),figsize=(2*len(bilder),2))
    for ax,img,txt in zip(np.atleast_1d(axs),bilder,titel):
        ax.imshow(img.reshape(8,8),cmap="gray",vmin=0,vmax=1);ax.set_title(str(txt));ax.axis("off")
    plt.show()
zeigen(X[test[:5]],digit_model.predict(X[test[:5]]))

## DL 5 · Buchstaben als unbekannte Eingaben (20–25 Min)
Das Netz kennt nur die Ausgaben **0 bis 9**. Wir zeichnen vereinfachte Buchstaben A und B als Pixelbilder. **Vermutet zuerst**, welche Ziffern das Netz ausgeben könnte.

Die Buchstabenbilder sind bewusst schematisch, keine echte Handschriftendatenbank.

In [ ]:
A_rows=["00111000","01000100","01000100","01111100","01000100","01000100","01000100","00000000"]
B_rows=["01111000","01000100","01000100","01111000","01000100","01000100","01111000","00000000"]
def raster(rows):
    return np.array([[int(c) for c in row] for row in rows],dtype=float)
A=raster(A_rows); B=raster(B_rows)
zeigen([A,B],["A","B"])
print("Ziffernnetz sagt für A und B:",digit_model.predict(np.array([A.ravel(),B.ravel()])))

**Diskussion:** Hat das Netz Buchstaben gelernt? Warum gibt es dennoch eine Ziffer aus? Notiert die Vorhersagen und eure Erklärung auf der Kursseite.

## DL 6 · Angeleitetes Fine-Tuning: neuer Klassifikationskopf (35 Min)
Wir **behalten die erste gelernte Schicht** des Ziffernnetzes und ersetzen seine Ziffernausgabe durch einen neuen Kopf für A/B. Nur dieser neue Kopf wird trainiert. Das ist **Transfer Learning mit eingefrorener Merkmalschicht**; nicht das vollständige Weitertrainieren aller Gewichte.

Für den Versuch erzeugen wir leicht verschobene und verrauschte Varianten. Das ist ein kontrolliertes Demonstrationsbeispiel, kein Beweis, dass das Verfahren für beliebige Fotos funktioniert.

In [ ]:
def varianten(basis,n,seed,noise=0.12):
    zufall=np.random.default_rng(seed)
    out=[]
    for _ in range(n):
        # leichte horizontale Verschiebung ohne zyklisches Überlaufen
        shift=int(zufall.integers(-1,2))
        img=np.zeros((8,8))
        if shift==0: img[:]=basis
        elif shift>0: img[:,shift:]=basis[:,:-shift]
        else: img[:,:shift]=basis[:,-shift:]
        out.append(np.clip(img+zufall.normal(0,noise,(8,8)),0,1).ravel())
    return np.array(out)
train_a=varianten(A,40,1);train_b=varianten(B,40,2)
test_a=varianten(A,20,101);test_b=varianten(B,20,102)
X_ab=np.vstack([train_a,train_b]);y_ab=np.array([0]*40+[1]*40)
X_ab_test=np.vstack([test_a,test_b]);y_ab_test=np.array([0]*20+[1]*20)
def merkmale(bilder):
    # Die Gewichte der ersten Schicht bleiben unverändert.
    return np.maximum(0,bilder@digit_model.coefs_[0]+digit_model.intercepts_[0])
kopf=LogisticRegression(max_iter=500,random_state=12)
kopf.fit(merkmale(X_ab),y_ab)
vorher=digit_model.predict(X_ab_test)  # nur Ziffern, keine A/B-Labels
nachher=kopf.predict(merkmale(X_ab_test))
print("Vorher: mögliche Ausgaben",sorted(set(vorher)),"(nur Ziffern)")
print("Nachher: A/B-Trefferquote",round(accuracy_score(y_ab_test,nachher),2))
zeigen(X_ab_test[[0,3,20,39]],["A","A","B","B"])

**Sicherung:** Warum dürfen wir eine Ziffernausgabe nicht direkt als «falsch erkannter Buchstabe» zählen? Was hat sich beim neuen Klassifikationskopf geändert? Vergleicht die **Art der Ausgabe** vor/nach dem Umbau, nicht zwei inkompatible Trefferquoten.

## DL 7 · Zwei eigene Bildklassen (35–45 Min)
Wählt zwei **eigene 8×8-Pixelmuster** (z. B. Pfeil und Kreuz). Ändert unten die beiden Raster. Trainiert auf Varianten eurer eigenen Muster einen neuen A/B-Kopf, während die gelernte erste Schicht eingefroren bleibt.

**Wichtig:** Das sind selbst entworfene Pixelbilder, keine Fotos. Für echte Gegenstandsfotos wäre ein anderes vortrainiertes Bildnetz nötig.

In [ ]:
klasse_0=[
"00011000",
"00111100",
"01111110",
"00011000",
"00011000",
"00011000",
"00011000",
"00000000"]
klasse_1=[
"10000001",
"01000010",
"00100100",
"00011000",
"00011000",
"00100100",
"01000010",
"10000001"]
eigen_0=raster(klasse_0);eigen_1=raster(klasse_1)
zeigen([eigen_0,eigen_1],["Klasse 0","Klasse 1"])
X_eigen=np.vstack([varianten(eigen_0,30,13),varianten(eigen_1,30,14)])
y_eigen=np.array([0]*30+[1]*30)
X_eigen_test=np.vstack([varianten(eigen_0,15,113),varianten(eigen_1,15,114)])
y_eigen_test=np.array([0]*15+[1]*15)
eigen_kopf=LogisticRegression(max_iter=500,random_state=12).fit(merkmale(X_eigen),y_eigen)
print("Eigene Klassen: neue Testbilder richtig:",
      sum(eigen_kopf.predict(merkmale(X_eigen_test))==y_eigen_test),"von",len(y_eigen_test))

### Fairer Vergleich
Verändert **nur eine** Bedingung: die Zahl der Trainingsvarianten pro Klasse. Trainiert erneut, aber verwendet **dieselben** Testbilder. Vergleicht die Ergebnisse und beschreibt eine Grenze eures Versuchs.

In [ ]:
anzahl=8 # Versucht z.B. 8 oder 30; sonst nichts ändern.
X_neu=np.vstack([varianten(eigen_0,anzahl,13),varianten(eigen_1,anzahl,14)])
y_neu=np.array([0]*anzahl+[1]*anzahl)
vergleich=LogisticRegression(max_iter=500,random_state=12).fit(merkmale(X_neu),y_neu)
for name,modell in [("Erstes Modell",eigen_kopf),("Vergleich",vergleich)]:
    pred=modell.predict(merkmale(X_eigen_test))
    print(name,[(int(np.sum(pred[y_eigen_test==k]==k)),int(np.sum(y_eigen_test==k))) for k in [0,1]])


**Vor dem Fachgespräch:** Speichert das ausgeführte Notebook. Erklärt den Unterschied zwischen Training und Test, was am Ziffernnetz übernommen wurde und welche Aussagen die wenigen Testbilder erlauben. Bei perfekter Trefferquote erfindet keine Fehler: Benennt stattdessen die begrenzte Aussagekraft.